# 08 — Multiple TPV Route Planning
Upper-bound feasibility analysis for multi-TPV survey missions.
Given N TPV shapefiles and a flight budget, estimates:
- Maximum number of TPVs observable in one flight
- Maximum total chord crossings for each feasible TPV subset
- Approximate route sketch (not optimised waypoints)
Satellite coincidence is treated as an independent stop (added after TPV selection).

In [ ]:
import pickle
import itertools
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from math import cos, sin, pi, sqrt, radians, degrees, atan2
from shapely.geometry import Polygon, LineString
import warnings
warnings.filterwarnings('ignore')

CACHE_PATH  = 'mission_cache.pkl'
SPACING_KM  = 100.0    # minimum inter-chord spacing (km)
MAX_CHORDS  = 4        # max chords per TPV to consider
N_ANGLE_DEV = 3        # number of chord angle variations
ANGLE_DEV   = 5.0      # ± degrees from PCA axis
TURN_FACTOR = 1.2      # safety multiplier on turn-penalty estimates

In [ ]:
with open(CACHE_PATH, 'rb') as f:
    cache = pickle.load(f)

P             = cache['params']
BUDGET        = P['TOTAL_BUDGET_KM']
AC_SPEED      = P['AIRCRAFT_SPEED_KMH']
TURN_PEN      = P['TURN_PENALTY_KM']
BASE          = np.asarray(cache['BASE'], float)
ORIGIN_LON, ORIGIN_LAT = cache['origin']

# Build numpy distance matrix from sp_full for fast lookups
sp_full      = cache['sp']
static_nodes = cache['static_nodes']
node_key     = cache['node_key']
n_static     = cache['n_static']
N_nodes      = n_static
D            = np.full((N_nodes, N_nodes), 1e18, dtype=np.float64)
for i in range(N_nodes):
    for j, v in sp_full[i].items():
        if j < N_nodes:
            D[i, j] = v
np.fill_diagonal(D, 0.0)
BASE_IDX = node_key.get(tuple(np.round(BASE, 3)))

def wgs84_to_km(lon, lat):
    x = (np.asarray(lon, float) - ORIGIN_LON) * cos(radians(ORIGIN_LAT)) * 111.32
    y = (np.asarray(lat, float) - ORIGIN_LAT) * 110.54
    return x, y

def fit_pca_ellipse(pts):
    pts = np.asarray(pts, float)
    center = pts.mean(axis=0)
    cov = np.cov((pts - center).T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
    a   = 2.0 * sqrt(max(eigvals[0], 0.))
    b   = 2.0 * sqrt(max(eigvals[1], 0.))
    phi = atan2(eigvecs[1, 0], eigvecs[0, 0])
    return center, a, b, phi

def ellipse_chord_len(b, offset, a):
    """Chord length across ellipse at given offset from centre (along major axis)."""
    ratio = offset / a if a > 1e-6 else 0.0
    return 2.0 * b * sqrt(max(0.0, 1.0 - ratio**2))

def snap_to_graph(pt):
    """Return index of nearest static node to pt."""
    dists = np.linalg.norm(static_nodes - pt, axis=1)
    return int(np.argmin(dists))

def sp_dist(i, j):
    """Shortest-path distance between node indices i and j."""
    if i is None or j is None:
        return 1e18
    return float(D[i, j])

print(f'Cache loaded: {N_nodes} static nodes, budget={BUDGET:.0f} km, '
      f'turn_penalty={TURN_PEN:.0f} km/turn')
print(f'BASE index: {BASE_IDX}, BASE coords: ({BASE[0]:.1f}, {BASE[1]:.1f}) km')

In [ ]:
def load_tpv(shp_path, label=None):
    """Load a TPV shapefile, fit PCA ellipse, return TPV descriptor dict."""
    gdf = gpd.read_file(shp_path)
    coords_wgs = np.array(gdf.geometry.iloc[0].coords)
    x, y = wgs84_to_km(coords_wgs[:, 0], coords_wgs[:, 1])
    pts = np.column_stack([x, y])
    center, a, b, phi = fit_pca_ellipse(pts)
    poly = Polygon(pts).buffer(0)
    
    # Nearest static node to TPV centre
    centre_idx = snap_to_graph(center)
    # Nearest static node to each ellipse edge (approximate)
    edge_near = center + (np.linalg.norm(center - BASE) > 1e-3) * \
                (BASE - center) / max(np.linalg.norm(BASE - center), 1e-6) * a
    edge_near_idx = snap_to_graph(edge_near)
    
    # Min transit: BASE → nearest TPV point (approx)
    d_base_to_centre = float(np.linalg.norm(center - BASE))
    d_base_to_edge   = max(0.0, d_base_to_centre - a)
    
    return dict(
        label    = label or shp_path,
        pts      = pts,
        poly     = poly,
        center   = center,
        a=a, b=b, phi=phi,
        d_base   = d_base_to_centre,
        d_edge   = d_base_to_edge,
        centre_idx = centre_idx,
        edge_near_idx = edge_near_idx,
    )

# ── Edit this list to load your TPV shapefiles ──────────────────────────────
TPV_PATHS = [
    ('C:/nasa-ffp-nurture/test_data/Contour_2_14_18_3.shp', 'TPV-A'),
    # ('C:/nasa-ffp-nurture/test_data/Contour_B.shp', 'TPV-B'),
    # Add more TPVs here
]

tpvs = []
for path, label in TPV_PATHS:
    try:
        tpv = load_tpv(path, label)
        tpvs.append(tpv)
        print(f'{label}: centre=({tpv["center"][0]:.0f},{tpv["center"][1]:.0f}) km  '
              f'a={tpv["a"]:.0f} b={tpv["b"]:.0f} km  d_base={tpv["d_base"]:.0f} km')
    except Exception as e:
        print(f'WARNING: could not load {path}: {e}')

print(f'\nLoaded {len(tpvs)} TPV(s)')

In [ ]:
def tpv_chord_cost_profile(tpv, n_max=MAX_CHORDS, spacing=SPACING_KM):
    """
    For each n in 1..n_max, estimate the survey cost (chord lengths + internal transitions)
    using ellipse approximation only — no Shapely polygon intersection needed.
    
    Returns list of (n_chords, survey_dist_km, n_chord_actual) where survey_dist_km
    is the distance while INSIDE the TPV (chord crossings only, not transit to/from BASE).
    """
    a, b = tpv['a'], tpv['b']
    results = []
    for n in range(1, n_max + 1):
        # n parallel chords spaced `spacing` km apart, centred on ellipse centre
        half_span = (n - 1) / 2.0 * spacing
        if half_span > a * 0.95:
            break  # chords don't fit inside ellipse
        offsets = np.linspace(-half_span, half_span, n)
        chord_lens = [ellipse_chord_len(b, d, a) for d in offsets]
        valid = [cl for cl in chord_lens if cl > 10.0]
        if not valid:
            break
        n_actual = len(valid)
        survey_dist = sum(valid) + (n_actual - 1) * spacing  # chords + inter-chord transit
        results.append((n_actual, survey_dist))
    return results

# Compute profiles for all TPVs
for tpv in tpvs:
    profile = tpv_chord_cost_profile(tpv)
    tpv['chord_profile'] = profile
    print(f'{tpv["label"]}:')
    for n, dist in profile:
        print(f'  {n} chord(s): survey_dist ≈ {dist:.0f} km  '
              f'(2×transit≈{2*tpv["d_edge"]:.0f} km, total≈{dist+2*tpv["d_edge"]:.0f} km)')

In [ ]:
def transit_between_tpvs(tpv_i, tpv_j):
    """Approximate transit distance between the closest edges of two TPVs."""
    # Use nearest static graph nodes to each TPV centre as proxies
    i_idx = tpv_i['centre_idx']
    j_idx = tpv_j['centre_idx']
    d = sp_dist(i_idx, j_idx)
    # Subtract the approximate radii (entering/exiting at near edges)
    d_adjusted = max(0.0, d - tpv_i['a'] * 0.5 - tpv_j['a'] * 0.5)
    return d_adjusted

N_tpv = len(tpvs)
T = np.zeros((N_tpv, N_tpv))
for i in range(N_tpv):
    for j in range(N_tpv):
        if i != j:
            T[i, j] = transit_between_tpvs(tpvs[i], tpvs[j])

print('Inter-TPV transit cost matrix (km):')
header = '        ' + '  '.join(f'{t["label"]:>8}' for t in tpvs)
print(header)
for i, tpvi in enumerate(tpvs):
    row = f'{tpvi["label"]:>8}  ' + '  '.join(f'{T[i,j]:8.0f}' for j in range(N_tpv))
    print(row)

In [ ]:
def greedy_tpv_selection(tpvs, T, base_idx, D, budget, turn_pen, spacing=SPACING_KM):
    """
    Greedy algorithm: repeatedly add the TPV with best (chords / marginal_cost) ratio
    until budget is exhausted. Returns list of (tpv_index, n_chords, survey_dist).
    
    Route structure: BASE → [TPV_1 survey] → [TPV_2 survey] → ... → BASE
    Transit between TPVs uses the static graph SP.
    """
    selected = []
    remaining_budget = budget
    current_pos_idx = base_idx
    visited = set()

    while True:
        best_gain = -1; best_tpv_i = None; best_n = 0; best_cost = 0

        for i, tpv in enumerate(tpvs):
            if i in visited:
                continue
            transit_in = sp_dist(current_pos_idx, tpv['centre_idx'])
            # Return cost: from this TPV back to BASE
            transit_out = sp_dist(tpv['centre_idx'], base_idx)
            
            for n, survey_dist in tpv['chord_profile']:
                # Marginal cost = transit_in + survey + turn penalties (rough)
                n_turns_est = n + 2  # enter TPV, cross n chords, exit
                turn_cost = n_turns_est * turn_pen * TURN_FACTOR
                # Budget check: can we still return to BASE after this TPV?
                marginal = transit_in + survey_dist + turn_cost
                if marginal + transit_out <= remaining_budget:
                    gain_per_cost = n / max(marginal, 1.0)
                    if gain_per_cost > best_gain:
                        best_gain = gain_per_cost
                        best_tpv_i = i
                        best_n = n
                        best_cost = marginal
                    break  # take max feasible n for this TPV (profile sorted ascending)

        if best_tpv_i is None:
            break  # no more TPVs fit

        tpv = tpvs[best_tpv_i]
        selected.append((best_tpv_i, best_n, best_cost))
        visited.add(best_tpv_i)
        remaining_budget -= best_cost
        current_pos_idx = tpv['centre_idx']

    return selected

result = greedy_tpv_selection(tpvs, T, BASE_IDX, D, BUDGET, TURN_PEN)

print(f'Greedy selection ({len(result)} TPV(s)):')
total_chords = 0
total_cost   = 0
for tpv_i, n_chords, cost in result:
    print(f'  {tpvs[tpv_i]["label"]}: {n_chords} chord(s), marginal cost ≈ {cost:.0f} km')
    total_chords += n_chords
    total_cost   += cost
print(f'\nTotal: {total_chords} chord crossings, estimated cost ≈ {total_cost:.0f} km '
      f'(budget {BUDGET:.0f} km, remaining ≈ {BUDGET - total_cost:.0f} km)')

In [ ]:
def enumerate_tpv_subsets(tpvs, T, base_idx, D, budget, turn_pen, max_subset=None):
    """
    Enumerate all subsets of TPVs and find the one maximising total chord crossings
    that fits within budget. Uses ellipse approximation for cost estimates.
    
    Returns list of (subset_indices, n_chords_per_tpv, total_chords, total_cost).
    """
    N = len(tpvs)
    if max_subset is None:
        max_subset = N
    
    best_total_chords = 0
    all_feasible = []

    for size in range(1, min(max_subset, N) + 1):
        for subset in itertools.combinations(range(N), size):
            # Try all orderings of this subset (small N only)
            if size <= 6:
                orderings = list(itertools.permutations(subset))
            else:
                orderings = [subset]  # too many orderings, just try one

            for order in orderings:
                cost = 0.0
                pos_idx = base_idx
                n_chords_each = []
                feasible = True

                for tpv_i in order:
                    tpv = tpvs[tpv_i]
                    transit = sp_dist(pos_idx, tpv['centre_idx'])
                    # Find max n_chords for remaining budget
                    ret_cost = sp_dist(tpv['centre_idx'], base_idx)
                    best_n = 0; best_survey = 0
                    for n, survey_dist in tpv['chord_profile']:
                        n_turns_est = (n + 2) * TURN_FACTOR
                        tpv_cost = transit + survey_dist + n_turns_est * turn_pen
                        if cost + tpv_cost + ret_cost <= budget:
                            best_n = n; best_survey = survey_dist
                    if best_n == 0:
                        feasible = False; break
                    n_turns_est = (best_n + 2) * TURN_FACTOR
                    cost += transit + best_survey + n_turns_est * turn_pen
                    n_chords_each.append(best_n)
                    pos_idx = tpv['centre_idx']

                if not feasible:
                    continue
                cost += sp_dist(pos_idx, base_idx)
                if cost > budget:
                    continue

                total_chords = sum(n_chords_each)
                all_feasible.append((list(order), n_chords_each, total_chords, cost))
                if total_chords > best_total_chords:
                    best_total_chords = total_chords

    # Sort by total chords descending, then cost ascending
    all_feasible.sort(key=lambda x: (-x[2], x[3]))
    return all_feasible

feasible_plans = enumerate_tpv_subsets(tpvs, T, BASE_IDX, D, BUDGET, TURN_PEN)

print(f'Feasible plans found: {len(feasible_plans)}')
print(f'\nTop 10 plans (by total chord crossings):')
for order, n_each, total, cost in feasible_plans[:10]:
    labels = ' → '.join(f'{tpvs[i]["label"]}(×{n})' for i, n in zip(order, n_each))
    print(f'  {total} chords | cost≈{cost:.0f} km | {labels}')

In [ ]:
def draw_ellipse_approx(ax, tpv, n_chords=1, spacing=SPACING_KM, color='#2266cc', alpha=0.3):
    """Draw TPV ellipse approximation and n_chords parallel chord lines."""
    center, a, b, phi = tpv['center'], tpv['a'], tpv['b'], tpv['phi']
    # Draw ellipse outline
    theta = np.linspace(0, 2*pi, 200)
    ex = center[0] + a * np.cos(theta) * cos(phi) - b * np.sin(theta) * sin(phi)
    ey = center[1] + a * np.cos(theta) * sin(phi) + b * np.sin(theta) * cos(phi)
    ax.fill(ex, ey, color=color, alpha=alpha, zorder=2)
    ax.plot(ex, ey, color=color, lw=1.5, zorder=3)
    # Draw chord lines
    major = np.array([cos(phi), sin(phi)])
    perp  = np.array([-sin(phi), cos(phi)])
    half_span = (n_chords - 1) / 2.0 * spacing
    offsets = np.linspace(-half_span, half_span, n_chords)
    for d in offsets:
        cl = ellipse_chord_len(b, d, a)
        if cl < 10: continue
        pt_c = center + d * major
        ax.plot([pt_c[0] - cl/2 * perp[0], pt_c[0] + cl/2 * perp[0]],
                [pt_c[1] - cl/2 * perp[1], pt_c[1] + cl/2 * perp[1]],
                color=color, lw=2, zorder=4, alpha=0.85)
    ax.annotate(tpv['label'], center, fontsize=9, fontweight='bold',
                ha='center', va='center', color=color, zorder=5)

fig, ax = plt.subplots(figsize=(13, 11))

# Draw satellite track
seg_pts  = cache['sat_track']['seg_pts']
ax.plot(seg_pts[:, 0], seg_pts[:, 1], color='gray', lw=1, ls='--',
        zorder=1, label='Satellite track', alpha=0.5)

# Draw restricted / ATC zones
def draw_shapely(geom, ax, **kw):
    if geom is None or geom.is_empty: return
    parts = list(geom.geoms) if hasattr(geom, 'geoms') else [geom]
    for p in parts:
        if hasattr(p, 'exterior'):
            x, y = p.exterior.xy; ax.fill(x, y, **kw)

draw_shapely(cache['restricted_union'], ax, color='#ff4444', alpha=0.2, label='Restricted')
draw_shapely(cache['atc_union'],        ax, color='#ff9900', alpha=0.12, label='ATC zone')
draw_shapely(cache['dropsonde_union'],  ax, color='#88cc88', alpha=0.12, label='Dropsonde')

# Draw BASE
ax.scatter(*BASE, s=200, marker='*', color='black', zorder=8, label='BASE (Goose Bay)')

# Draw best plan
colors = ['#2266cc', '#cc4400', '#228822', '#aa22aa', '#cc8800']
if feasible_plans:
    best_order, best_n_each, best_total, best_cost = feasible_plans[0]
    # Draw approximate route: BASE → TPV centroids → BASE
    route_pts = [BASE] + [tpvs[i]['center'] for i in best_order] + [BASE]
    route_arr = np.array(route_pts)
    ax.plot(route_arr[:, 0], route_arr[:, 1], 'k--', lw=1.5, alpha=0.5, zorder=3,
            label='Approx route (centroids)')
    for k, (tpv_i, n_chords) in enumerate(zip(best_order, best_n_each)):
        col = colors[k % len(colors)]
        draw_ellipse_approx(ax, tpvs[tpv_i], n_chords=n_chords, color=col)
    title = (f'Best plan: {best_total} chord crossings across {len(best_order)} TPV(s)\n'
             f'Est. cost ≈ {best_cost:.0f} km (budget {BUDGET:.0f} km)\n'
             + ' + '.join(f'{tpvs[i]["label"]}(×{n})' for i,n in zip(best_order, best_n_each)))
else:
    for k, tpv in enumerate(tpvs):
        draw_ellipse_approx(ax, tpv, n_chords=1, color=colors[k % len(colors)])
    title = 'No feasible plan found within budget'

ax.set_aspect('equal')
ax.set_xlabel('East (km)'); ax.set_ylabel('North (km)')
ax.set_title(title, fontsize=10)
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, lw=0.4, alpha=0.5)
plt.tight_layout()
plt.savefig('multi_tpv_upper_bound.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multi_tpv_upper_bound.png')

In [ ]:
# ── Satellite coincidence upper-bound estimate ──────────────────────────────
# After selecting the best multi-TPV subset, estimate the maximum satellite
# coincidence achievable by inserting one sat segment anywhere along the route.
#
# All pa/pb in sat_arc_variants are already static graph nodes, so D[i,j] is
# exact — no snapping needed.

sat_arc_variants = cache.get('sat_arc_variants', {})
sat_seg_pts      = np.asarray(cache['sat_track']['seg_pts'])
sat_cum_arc      = np.asarray(cache['sat_track']['cum_arc'])

print(f'sat_arc_variants: {len(sat_arc_variants)} entry positions in cache')


def estimate_sat_coincidence_ub(plan, tpvs, sat_arc_vars, D, node_key, budget, ac_speed):
    """Upper-bound estimate for satellite coincidence arc on a multi-TPV plan.

    For each satellite entry pa, the detour cost is:
      min_transit(route_node -> pa) + arc_km + min_transit(pb -> route_node)
    Returns (best_arc_km, best_pa_key, best_pb_arr) or None.
    """
    if not plan or not sat_arc_vars:
        return None
    order, n_each, total_chords, plan_cost = plan
    remaining = budget - plan_cost
    if remaining <= 0:
        return None

    route_idxs = [BASE_IDX] + [tpvs[i]['centre_idx'] for i in order]
    route_idxs = [ri for ri in route_idxs if ri is not None and ri < len(D)]

    best_arc = 0.0
    best_pa_key = best_pb_arr = None

    for pa_key, variants in sat_arc_vars.items():
        pa_idx = node_key.get(pa_key)
        if pa_idx is None or pa_idx >= len(D):
            continue
        min_to_pa = min(float(D[ri, pa_idx]) for ri in route_idxs)
        if min_to_pa >= 1e17:
            continue

        for pb_arr, arc_km in reversed(variants):   # largest arc first
            pb_key = tuple(np.round(np.asarray(pb_arr), 3))
            pb_idx = node_key.get(pb_key)
            if pb_idx is None or pb_idx >= len(D):
                continue
            min_from_pb = min(float(D[pb_idx, ri]) for ri in route_idxs)
            if min_from_pb >= 1e17:
                continue
            detour = min_to_pa + arc_km + min_from_pb
            if detour <= remaining + 1e-6 and arc_km > best_arc:
                best_arc    = arc_km
                best_pa_key = pa_key
                best_pb_arr = pb_arr
                break   # max feasible arc for this pa found

    return (best_arc, best_pa_key, best_pb_arr) if best_arc > 0 else None


print()
if not feasible_plans:
    print('No feasible multi-TPV plan — cannot estimate satellite coincidence.')
    _sat_ub = None
else:
    _best_plan  = feasible_plans[0]
    _order, _n_each, _total_chords, _plan_cost = _best_plan

    _sat_ub = estimate_sat_coincidence_ub(
        _best_plan, tpvs, sat_arc_variants, D, node_key, BUDGET, AC_SPEED)

    _plan_label = ' -> '.join(
        f'{tpvs[i]["label"]}(x{n})' for i, n in zip(_order, _n_each))
    print(f'Best multi-TPV plan  : {_plan_label}')
    print(f'Plan cost (estimate) : {_plan_cost:.0f} km'
          f'  |  budget {BUDGET:.0f} km  |  remaining {BUDGET - _plan_cost:.0f} km')
    print()

    if _sat_ub:
        _arc_km, _pa_key, _pb_arr = _sat_ub
        _arc_min = _arc_km / AC_SPEED * 60.0
        _pa_pt   = np.array(_pa_key, dtype=float)
        _pb_pt   = np.asarray(_pb_arr, dtype=float)
        print(f'Satellite coincidence upper bound:')
        print(f'  Max arc length : {_arc_km:.0f} km  ({_arc_min:.0f} min)')
        print(f'  Entry pa       : ({_pa_pt[0]:.0f}, {_pa_pt[1]:.0f}) km from BASE')
        print(f'  Exit  pb       : ({_pb_pt[0]:.0f}, {_pb_pt[1]:.0f}) km from BASE')
    else:
        print('No satellite coincidence fits within remaining budget.')

# ── Figure: multi-TPV route + satellite coincidence segment ──────────────────
fig, ax = plt.subplots(figsize=(13, 11))

ax.plot(sat_seg_pts[:, 0], sat_seg_pts[:, 1], '--', color='#8800bb',
        lw=1, alpha=0.35, zorder=1, label='Satellite track')

def _draw_zone(geom, ax, **kw):
    if geom is None or geom.is_empty:
        return
    parts = list(geom.geoms) if hasattr(geom, 'geoms') else [geom]
    for p in parts:
        if hasattr(p, 'exterior'):
            x, y = p.exterior.xy
            ax.fill(x, y, **kw)

_draw_zone(cache['restricted_union'], ax, color='#ff4444', alpha=0.20, zorder=2)
_draw_zone(cache['atc_union'],        ax, color='#ff9900', alpha=0.12, zorder=2)
_draw_zone(cache['dropsonde_union'],  ax, color='#88cc88', alpha=0.12, zorder=2)

ax.scatter(*BASE, s=200, marker='*', color='black', zorder=9, label='BASE (Goose Bay)')

_colors = ['#2266cc', '#cc4400', '#228822', '#aa22aa', '#cc8800']
if feasible_plans:
    _route_pts = np.array([BASE] + [tpvs[i]['center'] for i in _order] + [BASE])
    ax.plot(_route_pts[:, 0], _route_pts[:, 1], 'k--', lw=1.5, alpha=0.5,
            zorder=3, label='Approx route (centroids)')
    for k, (tpv_i, n_chords) in enumerate(zip(_order, _n_each)):
        draw_ellipse_approx(ax, tpvs[tpv_i], n_chords=n_chords,
                            color=_colors[k % len(_colors)])

if _sat_ub:
    _arc_km, _pa_key, _pb_arr = _sat_ub
    _arc_min = _arc_km / AC_SPEED * 60.0
    _pa_pt   = np.array(_pa_key, dtype=float)
    _pb_pt   = np.asarray(_pb_arr, dtype=float)
    # Draw coincidence arc along satellite track
    _d_pa = np.linalg.norm(sat_seg_pts - _pa_pt, axis=1)
    _d_pb = np.linalg.norm(sat_seg_pts - _pb_pt, axis=1)
    _i_pa, _i_pb = int(np.argmin(_d_pa)), int(np.argmin(_d_pb))
    if _i_pa > _i_pb:
        _i_pa, _i_pb = _i_pb, _i_pa
    _arc_track = sat_seg_pts[_i_pa:_i_pb + 1]
    ax.plot(_arc_track[:, 0], _arc_track[:, 1], '-', color='#8800bb',
            lw=5, zorder=8, solid_capstyle='round',
            label=f'Best coincidence ~{_arc_km:.0f} km ({_arc_min:.0f} min) [upper bound]')
    ax.scatter(*_pa_pt, s=140, color='#8800bb', marker='o', zorder=10)
    ax.scatter(*_pb_pt, s=140, color='#8800bb', marker='s', zorder=10)
    ax.annotate('pa', _pa_pt, textcoords='offset points', xytext=(6, 4),
                fontsize=8, color='#8800bb', fontweight='bold')
    ax.annotate('pb', _pb_pt, textcoords='offset points', xytext=(6, 4),
                fontsize=8, color='#8800bb', fontweight='bold')

ax.set_aspect('equal')
ax.set_xlabel('East (km)'); ax.set_ylabel('North (km)')
_title = ['Multi-TPV plan + satellite coincidence upper bound']
if feasible_plans:
    _title.append(_plan_label + f'  |  plan cost ~{_plan_cost:.0f} km')
if _sat_ub:
    _title.append(f'Max coincidence ~{_arc_km:.0f} km  ({_arc_min:.0f} min)')
ax.set_title('\n'.join(_title), fontsize=10)
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, lw=0.4, alpha=0.5)
plt.tight_layout()
plt.savefig('multi_tpv_sat_coincidence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multi_tpv_sat_coincidence.png')


In [ ]:
print('='*70)
print(f'MULTI-TPV UPPER BOUND SUMMARY')
print(f'Budget: {BUDGET:.0f} km  |  Turn penalty: {TURN_PEN:.0f} km/turn')
print(f'TPVs loaded: {len(tpvs)}')
print('='*70)
for tpv in tpvs:
    print(f'\n{tpv["label"]}:')
    print(f'  Centre: ({tpv["center"][0]:.0f}, {tpv["center"][1]:.0f}) km')
    print(f'  Ellipse: a={tpv["a"]:.0f} km, b={tpv["b"]:.0f} km, phi={degrees(tpv["phi"]):.1f}°')
    print(f'  Min transit from BASE: ≈ {tpv["d_edge"]:.0f} km (edge), ≈ {tpv["d_base"]:.0f} km (centre)')
    print(f'  Chord profile:')
    for n, dist in tpv['chord_profile']:
        print(f'    n={n}: survey≈{dist:.0f} km, roundtrip≈{dist+2*tpv["d_edge"]:.0f} km')

print(f'\n{"="*70}')
print(f'Top 5 feasible plans:')
for rank, (order, n_each, total, cost) in enumerate(feasible_plans[:5], 1):
    labels = ' → '.join(f'{tpvs[i]["label"]}(×{n})' for i, n in zip(order, n_each))
    print(f'  #{rank}: {total} chords | cost≈{cost:.0f}/{BUDGET:.0f} km | {labels}')
if not feasible_plans:
    print('  No feasible plans found.')
print('='*70)